# **LIMPIEZA DEL CATÁLOGO SÍSMICO DEL IGP (1960-2026)**

**Proyecto:** Análisis sísmico del Perú

**Fuente:** Centro Sismológico Nacional (CENSIS) – Instituto Geofísico del Perú (IGP). Catálogo instrumental descargado de https://ultimosismo.igp.gob.pe/repositorio/datos-sismicos (1960 – 11/10/2026).

Este notebook toma el catálogo original del IGP, lo limpia paso a paso y genera `sismos_limpio.csv`, que es el archivo que usará el dashboard en Power BI.

## **1. LIBRERÍAS**

In [1]:
import numpy as np
import pandas as pd
import json
from matplotlib.path import Path as Poligono

## **2. CARGA DE DATOS**

El catálogo se descargó en formato Excel (.xlsx), Los nombres originales de las columnas tienen espacios y símbolos, así que se renombran a nombres simples en mayúsculas.

In [2]:
df = pd.read_excel("IGP_catalogo_instrumental_1960_2026.xlsx")
print("Columnas originales:", list(df.columns))

df.columns = ["FECHA_UTC", "HORA_UTC", "LATITUD", "LONGITUD", "PROFUNDIDAD", "MAGNITUD"]
print("Filas:", len(df))
df.head()

Columnas originales: ['fecha UTC', 'hora UTC', 'latitud (º)', 'longitud (º)', 'profundidad (km)', 'magnitud (M)']
Filas: 25797


,FECHA_UTC,HORA_UTC,LATITUD,LONGITUD,PROFUNDIDAD,MAGNITUD
0,1960-01-13,15:40:34.00,-16.145,-72.144,60,7.5
1,1960-01-15,09:30:24.00,-15.000,-75.000,70,7.0
2,1960-01-17,02:57:58.00,-14.500,-74.500,150,6.4
3,1960-01-23,03:37:32.00,-12.500,-68.500,300,5.8
4,1960-01-30,05:07:24.00,-5.500,-77.500,100,5.7


## **3. EXPLORACIÓN INICIAL**

En esta sección solo se revisa el dataset para encontrar problemas.

### 3.1 Estructura y tipos

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25797 entries, 0 to 25796
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FECHA_UTC    25797 non-null  str    
 1   HORA_UTC     25797 non-null  str    
 2   LATITUD      25797 non-null  float64
 3   LONGITUD     25797 non-null  float64
 4   PROFUNDIDAD  25797 non-null  int64  
 5   MAGNITUD     25797 non-null  float64
dtypes: float64(3), int64(1), str(2)
memory usage: 1.2 MB


No hay nulos. Las columnas FECHA_UTC y HORA_UTC vienen como texto, por lo que hay que convertirlas a fecha real.

### 3.2 Estadísticas

In [4]:
df.describe().round(2)

,LATITUD,LONGITUD,PROFUNDIDAD,MAGNITUD
count,25797.00,25797.00,25797.00,25797.00
mean,-11.40,-75.69,70.34,4.67
std,4.48,3.09,65.24,0.48
min,-23.40,-82.89,5.00,3.00
25%,-15.40,-77.73,29.00,4.50
50%,-11.80,-75.72,48.00,4.60
75%,-7.97,-73.60,105.00,4.90
max,-1.49,-66.98,743.00,8.40


La latitud llega hasta -23.4°, más al sur que el Perú (≈ -18.4°): el catálogo incluye sismos de países vecinos.

### 3.3 Formato de la hora

Se cuenta cuántos caracteres tiene cada hora para ver si todas tienen el mismo formato.

In [5]:
largo = df["HORA_UTC"].str.len()
print(largo.value_counts().sort_index())

# Un ejemplo de cada formato y el rango de fechas en que aparece
df.groupby(largo).agg(ejemplo=("HORA_UTC", "first"), desde=("FECHA_UTC", "min"), hasta=("FECHA_UTC", "max"))

HORA_UTC
8      4726
9         3
11    21068
Name: count, dtype: int64


,ejemplo,desde,hasta
HORA_UTC,,,
8,15:54:05,2020-01-01,2026-10-11
9,21:23:01,2022-09-06,2022-09-09
11,15:40:34.00,1960-01-13,2019-12-31


Los 3 registros de 9 caracteres: con repr se ve el espacio al final.

In [6]:
df.loc[largo == 9, ["FECHA_UTC", "HORA_UTC"]].assign(HORA_VISIBLE=lambda x: x["HORA_UTC"].map(repr))

,FECHA_UTC,HORA_UTC,HORA_VISIBLE
23018,2022-09-06,21:23:01,'21:23:01 '
23019,2022-09-07,21:57:34,'21:57:34 '
23022,2022-09-09,21:59:41,'21:59:41 '


Problema encontrado: la hora viene en 2 formatos:

15:40:34.00 (11 caracteres, con centésimas de segundo) desde 1960 hasta 2019.

13:43:01 (8 caracteres) desde 2020.

Además, 3 registros de 2022 tienen un espacio al final.

### 3.4 Sismos duplicados

Un sismo es único por su momento, lugar, profundidad y magnitud.

In [7]:
columnas = ["FECHA_UTC", "HORA_UTC", "LATITUD", "LONGITUD", "PROFUNDIDAD", "MAGNITUD"]
print("Duplicados:", df.duplicated(columnas).sum())

df[df.duplicated(columnas, keep=False)]   # vemos cuáles son

Duplicados: 0


,FECHA_UTC,HORA_UTC,LATITUD,LONGITUD,PROFUNDIDAD,MAGNITUD


No hay sismos duplicados.

### 3.5 Identificador (ID)

In [8]:
print("Columnas:", list(df.columns))
print("¿Tiene ID?:", "ID" in df.columns)

Columnas: ['FECHA_UTC', 'HORA_UTC', 'LATITUD', 'LONGITUD', 'PROFUNDIDAD', 'MAGNITUD']
¿Tiene ID?: False


Este catálogo no incluye una columna de identificador (ID).

## **4. TRANSFORMACIÓN**

En esta sección se corrigen los problemas encontrados en la exploración.

### 4.1 Fecha y hora en UTC

Se quitan los espacios y se toman solo los primeros 8 caracteres hh:mm:ss, descartando las centésimas .00. Así todas las horas quedan con el mismo formato.

In [9]:
hora_texto = df["HORA_UTC"].str.strip().str[:8]
hora_texto.str.len().value_counts()    # todas deben tener 8 caracteres

HORA_UTC
8    25797
Name: count, dtype: int64

In [10]:
df["FECHA_HORA_UTC"] = pd.to_datetime(df["FECHA_UTC"] + " " + hora_texto,
                                      format="%Y-%m-%d %H:%M:%S", errors="coerce")

df["FECHA_HORA_UTC"].isna().sum()     # vemos si alguna fecha ha fallado

np.int64(0)

El diccionario del IGP indica que las fechas están en UTC. Perú está en UTC−5 y no usa horario de verano, así que basta restar 5 horas para obtener la hora local en la que ocurrió el sismo.

In [11]:
df["FECHA_HORA_LOCAL"] = df["FECHA_HORA_UTC"] - pd.Timedelta(hours=5)
df[["FECHA_UTC", "HORA_UTC", "FECHA_HORA_UTC", "FECHA_HORA_LOCAL"]].head(3)

,FECHA_UTC,HORA_UTC,FECHA_HORA_UTC,FECHA_HORA_LOCAL
0,1960-01-13,15:40:34.00,1960-01-13 15:40:34,1960-01-13 10:40:34
1,1960-01-15,09:30:24.00,1960-01-15 09:30:24,1960-01-15 04:30:24
2,1960-01-17,02:57:58.00,1960-01-17 02:57:58,1960-01-16 21:57:58


### 4.2 Eliminación de duplicados

In [12]:
df = df.drop_duplicates(columnas)
len(df)

25797

No se eliminó ninguna fila, no había duplicados.

### 4.3 Orden cronológico y nuevo ID

¿Está ordenado por fecha? Se cuentan las filas cuya fecha es anterior a la de la fila de arriba.

In [13]:
(df["FECHA_HORA_UTC"].diff().dt.total_seconds() < 0).sum()

np.int64(0)

El catálogo ya viene ordenado, 0 filas fuera de orden. Se ordena igual por seguridad y se crea un **ID consecutivo** (1, 2, 3, etc) como primera columna.

In [14]:
df = df.sort_values("FECHA_HORA_UTC").reset_index(drop=True)
df.insert(0, "ID", np.arange(1, len(df) + 1))
df.head(3)

,ID,FECHA_UTC,HORA_UTC,LATITUD,LONGITUD,PROFUNDIDAD,MAGNITUD,FECHA_HORA_UTC,FECHA_HORA_LOCAL
0,1,1960-01-13,15:40:34.00,-16.145,-72.144,60,7.5,1960-01-13 15:40:34,1960-01-13 10:40:34
1,2,1960-01-15,09:30:24.00,-15.000,-75.000,70,7.0,1960-01-15 09:30:24,1960-01-15 04:30:24
2,3,1960-01-17,02:57:58.00,-14.500,-74.500,150,6.4,1960-01-17 02:57:58,1960-01-16 21:57:58


### 4.4 Eliminación de columnas

Se eliminan FECHA_UTC y HORA_UTC porque ya están dentro de FECHA_HORA_UTC.

In [15]:
df = df.drop(columns=["FECHA_UTC", "HORA_UTC"])
list(df.columns)

['ID',
 'LATITUD',
 'LONGITUD',
 'PROFUNDIDAD',
 'MAGNITUD',
 'FECHA_HORA_UTC',
 'FECHA_HORA_LOCAL']

## **5. CALIDAD DE LOS DATOS**

En esta sección se buscan problemas que no se pueden corregir y se decide cómo manejarlos.

### 5.1 Huecos en el tiempo

Vemos cuántos sismos hay por año en los últimos años.

In [16]:
df["FECHA_HORA_LOCAL"].dt.year.value_counts().sort_index().tail(8)

FECHA_HORA_LOCAL
2019    565
2020    809
2021    834
2022    496
2023    463
2024    557
2025    839
2026    731
Name: count, dtype: int64

Revisemos de manera más detallada: meses sin ningún sismo registrado.

In [17]:
meses = df.set_index("FECHA_HORA_LOCAL").resample("MS").size()
meses[meses == 0]

FECHA_HORA_LOCAL
1960-06-01    0
1960-08-01    0
1960-10-01    0
1961-05-01    0
1962-05-01    0
1962-06-01    0
dtype: int64

El catálogo tiene datos continuos hasta el 11/10/2026. Solo hay 6 meses sin registros, entre 1960 y 1962, probablemente reales por la baja capacidad de detección de esa época.

### 5.2 Verificación de profundidad

In [18]:
print("Sismos con profundidad 0 km:", (df["PROFUNDIDAD"] == 0).sum())
print("Profundidad mínima:", df["PROFUNDIDAD"].min(), "km")

Sismos con profundidad 0 km: 0
Profundidad mínima: 5 km


En este catálogo no hay ninguna profundidad de 0 km

### 5.3 Magnitud de completitud (Mc)

Vemos, para cada época, la magnitud mínima registrada y la más frecuente.

In [19]:
df["ANIO"] = df["FECHA_HORA_LOCAL"].dt.year
for a, b in [(1960, 1980), (1980, 2000), (2000, 2010), (2010, 2018), (2018, 2027)]:
    m = df[(df["ANIO"] >= a) & (df["ANIO"] < b)]["MAGNITUD"]
    print(a, "-", b - 1, "| magnitud mínima:", m.min(), "| la más frecuente:", m.value_counts().idxmax())

1960 - 1979 | magnitud mínima: 4.0 | la más frecuente: 4.9
1980 - 1999 | magnitud mínima: 4.0 | la más frecuente: 4.5
2000 - 2009 | magnitud mínima: 4.0 | la más frecuente: 4.5
2010 - 2017 | magnitud mínima: 4.0 | la más frecuente: 4.5
2018 - 2026 | magnitud mínima: 3.0 | la más frecuente: 4.0


Creamos las columnas de control:

In [20]:
def mc(anio):
    if anio < 1980: return 4.9
    if anio < 2018: return 4.5
    return 4.0

df["MC_PERIODO"] = df["ANIO"].apply(mc)
df["SOBRE_MC"] = df["MAGNITUD"] >= df["MC_PERIODO"]
df["COMPARABLE_1960_2026"] = df["MAGNITUD"] >= 5.0

Sismos por década separados por tamaño.

In [21]:
tabla = df.groupby(df["ANIO"] // 10 * 10)["COMPARABLE_1960_2026"].value_counts().unstack()
tabla.columns = ["Menores a M5", "M5 o más"]
tabla.index.name = "Década"
tabla

,Menores a M5,M5 o más
Década,,
1960,554,448
1970,383,743
1980,1092,761
1990,3737,859
2000,3513,1015
2010,6916,1047
2020,4401,328


Si se cuentan todos los sismos por década, el total se multiplica por 8 entre los años 60 y los 2010, lo que haría pensar que cada vez tiembla más. Sin embargo, al separar por tamaño, los sismos de M5 o más se mantienen relativamente estables (entre 743 y 1,047 por década desde los 70), mientras que los menores a M5 pasan de 383 a 6,916.

Esto se debe a la mejora de la red sismográfica: con más estaciones, el IGP comenzó a registrar sismos pequeños que antes ocurrían pero no se detectaban. La prueba es la magnitud más frecuente de cada época, que baja de 4.9 (1960–1979) a 4.5 (1980–2017) y a 4.0 (2018–2026). A esto se le llama **sesgo de detección**. La década del 2020 está incompleta (2020 – octubre de 2026).

Para evitar conclusiones erróneas se crearon dos columnas de control:
- **SOBRE_MC:** marca los sismos cuya magnitud supera la magnitud de completitud (Mc) de su época, es decir, los que la red de ese momento podía registrar de forma completa.
- **COMPARABLE_1960_2026:** marca los sismos de magnitud M ≥ 5.0, un umbral fijo que cualquier red habría detectado desde 1960. Este filtro se usará para comparar la actividad sísmica entre épocas.

Revisemos los sismos M ≥ 5 por año en los últimos años:

In [22]:
df[df["COMPARABLE_1960_2026"]].groupby("ANIO").size().loc[2014:]

ANIO
2014    118
2015    124
2016    147
2017    129
2018    103
2019     39
2020     46
2021     46
2022     60
2023     39
2024     53
2025     51
2026     33
dtype: int64

Desde 2019 los sismos M ≥ 5 caen a cerca de un tercio (de ~125 a ~45 por año). Un cambio tan brusco probablemente se debe a un **cambio en el método de cálculo de la magnitud** del IGP, no a una disminución real de los sismos.

## **6. VARIABLES DERIVADAS Y CLASIFICACIONES**

Las variables derivadas (mes, día, hora y década) se generan en Python para centralizar la transformación de datos en un solo proceso. Power BI se usará exclusivamente para la visualización. Se calculan desde la **hora local**, para que cada sismo quede en el día y año en que ocurrió en el Perú.

**Clasificación por profundidad:** se basa en Hernando Tavera, presidente del IGP, quien explica que los sismos superficiales llegan hasta unos 60 km en la costa, los intermedios hasta 300 km y los profundos están entre 500 y 700 km. Él lo presenta como una descripción práctica del comportamiento sísmico peruano, no como una regla formal de corte. El estándar internacional (USGS) usa 70 km como límite de los superficiales.

En este proyecto se considera profundo todo sismo de más de 300 km, para que ningún sismo quede sin clasificar. Tavera no describe el tramo de 301 a 499 km, pero en este catálogo solo 5 sismos caen ahí entre 321 y 488 km, frente a 143 de 500 km o más, por lo que esta decisión no afecta los resultados.

In [23]:
t = df["FECHA_HORA_LOCAL"]
df["MES"] = t.dt.month
df["DIA"] = t.dt.day
df["HORA"] = t.dt.hour
df["DECADA"] = (df["ANIO"] // 10 * 10).astype(str) + "s"

df["TIPO_PROFUNDIDAD"] = pd.cut(df["PROFUNDIDAD"], [-1, 60, 300, 1000],
    labels=["Superficial (0-60 km)", "Intermedio (61-300 km)", "Profundo (>300 km)"])

df["RANGO_MAGNITUD"] = pd.cut(df["MAGNITUD"], [0, 3.9, 4.9, 5.9, 6.9, 10],
    labels=["< 4.0", "4.0 - 4.9", "5.0 - 5.9", "6.0 - 6.9", "7.0 o más"])

print(df["TIPO_PROFUNDIDAD"].value_counts(), "\n")
print(df["RANGO_MAGNITUD"].value_counts().sort_index())

TIPO_PROFUNDIDAD
Superficial (0-60 km)     15738
Intermedio (61-300 km)     9911
Profundo (>300 km)          148
Name: count, dtype: int64 

RANGO_MAGNITUD
< 4.0         1484
4.0 - 4.9    19112
5.0 - 5.9     4887
6.0 - 6.9      274
7.0 o más       40
Name: count, dtype: int64


## **7. UBICACIÓN DE LOS SISMOS**

El dataset solo trae coordenadas (latitud y longitud). En esta sección se calcula:
- La distancia a Lima de cada sismo.
- El departamento donde ocurrió, cruzando sus coordenadas con los límites de los departamentos del repositorio *peru-geojson* de GitHub (licencia libre MPL 2.0).
- La distancia a la costa de los sismos en el mar.

Cada sismo se clasifica en:
- **Tierra:** el epicentro está dentro de un departamento.
- **Mar:** el epicentro está en el océano; se asigna el departamento costero más cercano.
- **Fuera del Perú:** el epicentro está en un país vecino (Ecuador, Brasil, Bolivia o Chile).

### 7.1 Distancia a Lima

Distancia de cada sismo a Lima (Plaza de Armas) con la **fórmula de Haversine**, que mide distancias sobre la superficie curva de la Tierra (6371 = radio de la Tierra en km).

In [24]:
LIMA = (-12.0464, -77.0428)

def distancia_km(lat, lon, punto):
    la, lo = np.radians(punto[0]), np.radians(punto[1])
    lat, lon = np.radians(lat), np.radians(lon)
    a = np.sin((lat - la) / 2)**2 + np.cos(la) * np.cos(lat) * np.sin((lon - lo) / 2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["DIST_LIMA_KM"] = distancia_km(df["LATITUD"], df["LONGITUD"], LIMA).round(1)

# Sismos más fuertes a menos de 150 km de Lima
df[df["DIST_LIMA_KM"] <= 150].nlargest(5, "MAGNITUD")[["FECHA_HORA_LOCAL", "MAGNITUD", "PROFUNDIDAD", "DIST_LIMA_KM"]]

,FECHA_HORA_LOCAL,MAGNITUD,PROFUNDIDAD,DIST_LIMA_KM
1626,1974-10-03 09:21:34,7.7,21,59.5
1696,1974-11-09 07:59:55,7.2,13,79.6
1539,1974-01-05 03:33:51,6.6,92,90.6
5281,1993-04-18 04:16:23,6.4,107,49.3
827,1968-09-28 08:53:37,6.2,68,145.3


Según el catálogo del IGP, los últimos sismos de magnitud 7 o más a menos de 150 km de Lima ocurrieron en 1974, el terremoto del 3 de octubre (M7.7 según el IGP; Mw 8.1 según otras fuentes, a unos 60 km de Lima) y su réplica más fuerte, el 9 de noviembre (M7.2, a 79.6 km). Desde entonces, ningún sismo de magnitud 7 o más ha ocurrido dentro de ese radio. El terremoto de Pisco de 2007 (M8.0) quedó a 168 km, fuera del radio de 150 km.

Esto es consistente con lo que advierte el IGP: la costa central acumula energía desde el gran terremoto de 1746 (magnitud 8.8–9), por lo que se espera un sismo mayor a magnitud 8 en la zona.

El IGP no habla de un silencio sísmico desde 1974, sino desde 1746. Hernando Tavera explica que el último gran terremoto de la costa central fue el de 1746 (magnitud 8.8 a 9) y que desde entonces la zona acumula energía, por lo que se espera un sismo mayor a magnitud 8. Los simulacros del IGP simulan un M8.8 frente a Lima (Gestión). El sismo de 1974 fue fuerte, pero no liberó toda esa energía.

### 7.2 Departamentos

In [25]:
# Mapa de departamentos
with open("mapas/peru_departamental_simple.geojson", encoding="utf-8") as f:
    mapa = json.load(f)

puntos = df[["LONGITUD", "LATITUD"]].to_numpy()   # (x, y) de cada sismo
df["DEPARTAMENTO"] = None
vertices, vertices_dep = [], []                   # bordes de cada departamento

# Departamento que cae cada sismo?
for dep in mapa["features"]:
    nombre = dep["properties"]["NOMBDEP"]
    geom = dep["geometry"]
    partes = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    for parte in partes:
        borde = np.array(parte[0])[:, :2]
        dentro = Poligono(borde).contains_points(puntos)
        df.loc[dentro & df["DEPARTAMENTO"].isna(), "DEPARTAMENTO"] = nombre
        vertices.append(borde)
        vertices_dep += [nombre] * len(borde)

vertices = np.vstack(vertices)
vertices_dep = np.array(vertices_dep)
df["UBICACION"] = np.where(df["DEPARTAMENTO"].isna(), "Fuera del Perú", "Tierra")
print("Sismos dentro de un departamento:", df["DEPARTAMENTO"].notna().sum())
print("Sismos sin departamento (mar o fuera del Perú):", df["DEPARTAMENTO"].isna().sum())

Sismos dentro de un departamento: 15150
Sismos sin departamento (mar o fuera del Perú): 10647


### 7.3 Sismos en el mar

Se consideran en el mar si están entre las latitudes del Perú (-18.4° a -3.35°) y al oeste de la costa; a esos se les asigna el departamento costero más cercano. Funciona porque los países vecinos están al norte, este o sur del Perú, nunca al oeste. El resto queda como "Fuera del Perú".

In [26]:
COSTEROS = ["TUMBES", "PIURA", "LAMBAYEQUE", "LA LIBERTAD", "ANCASH", "LIMA",
            "CALLAO", "ICA", "AREQUIPA", "MOQUEGUA", "TACNA"]
es_costero = np.isin(vertices_dep, COSTEROS)
costa_xy, costa_dep = vertices[es_costero], vertices_dep[es_costero]

for i in np.where(df["DEPARTAMENTO"].isna())[0]:
    lon, lat = puntos[i]
    if not (-18.4 < lat < -3.35):                              
        continue
    franja = vertices[np.abs(vertices[:, 1] - lat) < 0.25]    
    if len(franja) and lon < franja[:, 0].min() + 0.6:        
        d = ((costa_xy[:, 0] - lon) * np.cos(np.radians(lat)))**2 + (costa_xy[:, 1] - lat)**2
        df.at[i, "DEPARTAMENTO"] = costa_dep[d.argmin()]       
        df.at[i, "UBICACION"] = "Mar"

df["DEPARTAMENTO"] = df["DEPARTAMENTO"].fillna("FUERA DEL PERÚ")
print(df["UBICACION"].value_counts(), "\n")

# Verificación con sismos conocidos
df.nlargest(8, "MAGNITUD")[["FECHA_HORA_LOCAL", "MAGNITUD", "DEPARTAMENTO", "UBICACION"]]

UBICACION
Tierra            15150
Mar                8957
Fuera del Perú     1690
Name: count, dtype: int64 



,FECHA_HORA_LOCAL,MAGNITUD,DEPARTAMENTO,UBICACION
9252,2001-06-23 15:33:14,8.4,AREQUIPA,Mar
478,1966-10-17 16:41:58,8.0,ANCASH,Mar
12169,2007-08-15 18:40:53,8.0,ICA,Mar
20732,2019-05-26 02:41:16,8.0,LORETO,Tierra
116,1963-08-15 12:25:06,7.8,PUNO,Tierra
1037,1970-05-31 15:23:32,7.8,ANCASH,Mar
1626,1974-10-03 09:21:34,7.7,CALLAO,Mar
6917,1996-11-12 11:59:40,7.7,ICA,Mar


La verificación coincide con la historia, Arequipa (Mar) 2001, Pisco (Mar) 2007 y Loreto (Tierra) 2019

### 7.4 Distancia a la costa

Para los sismos en el mar se calcula la distancia a la costa más cercana, usando los bordes de los departamentos costeros del GeoJSON. Permite distinguir un sismo frente a la costa de uno mar adentro. Para los sismos en tierra o fuera del Perú queda en 0.

In [27]:
df["DIST_COSTA_KM"] = 0.0
mar = df["UBICACION"] == "Mar"
for i in np.where(mar)[0]:
    lat, lon = df.at[i, "LATITUD"], df.at[i, "LONGITUD"]
    d = distancia_km(costa_xy[:, 1], costa_xy[:, 0], (lat, lon))
    df.at[i, "DIST_COSTA_KM"] = d.min().round(1)

df.loc[mar, "DIST_COSTA_KM"].describe().round(1)

count    8957.0
mean       60.0
std        47.1
min         0.2
25%        25.5
50%        49.5
75%        80.5
max       336.9
Name: DIST_COSTA_KM, dtype: float64

La mitad de los sismos marinos está a menos de 50 km de la costa y el más lejano a 337 km. Todos están dentro del dominio marítimo peruano (200 millas náuticas ≈ 370 km). La asignación al departamento costero es una aproximación, indica frente a qué departamento ocurrió el sismo.

## **8. VALIDACIÓN Y GUARDADO**

### 8.1 Validación

In [28]:
print("Filas:", len(df))
print("Vacíos en total:", df.isna().sum().sum())
print("Duplicados:", df.duplicated(["FECHA_HORA_UTC", "LATITUD", "LONGITUD"]).sum())
print("¿Ordenado por fecha?:", df["FECHA_HORA_UTC"].is_monotonic_increasing)
print("Rango de fechas:", df["FECHA_HORA_LOCAL"].min(), "→", df["FECHA_HORA_LOCAL"].max())
print("Magnitud:", df["MAGNITUD"].min(), "a", df["MAGNITUD"].max())
print("Profundidad:", df["PROFUNDIDAD"].min(), "a", df["PROFUNDIDAD"].max(), "km")
print("\n", df["UBICACION"].value_counts())

Filas: 25797
Vacíos en total: 0
Duplicados: 0
¿Ordenado por fecha?: True
Rango de fechas: 1960-01-13 10:40:34 → 2026-10-11 17:44:46
Magnitud: 3.0 a 8.4
Profundidad: 5 a 743 km

 UBICACION
Tierra            15150
Mar                8957
Fuera del Perú     1690
Name: count, dtype: int64


### 8.2 Guardado

In [29]:
orden = ["ID", "FECHA_HORA_UTC", "FECHA_HORA_LOCAL", "ANIO", "MES", "DIA", "HORA", "DECADA",
         "LATITUD", "LONGITUD", "PROFUNDIDAD", "TIPO_PROFUNDIDAD",
         "MAGNITUD", "RANGO_MAGNITUD", "MC_PERIODO", "SOBRE_MC", "COMPARABLE_1960_2026",
         "DEPARTAMENTO", "UBICACION", "DIST_LIMA_KM", "DIST_COSTA_KM"]

df[orden].to_csv("sismos_limpio.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: sismos_limpio.csv con {len(df):,} filas y {len(orden)} columnas")

Guardado: sismos_limpio.csv con 25,797 filas y 21 columnas


### 8.3 Comprobación

In [30]:
limpio = pd.read_csv("sismos_limpio.csv")
print("Filas:", len(limpio), "| Columnas:", limpio.shape[1])
print("Vacíos:", limpio.isna().sum().sum())
limpio.head()

Filas: 25797 | Columnas: 21
Vacíos: 0


,ID,FECHA_HORA_UTC,FECHA_HORA_LOCAL,ANIO,MES,DIA,HORA,DECADA,LATITUD,LONGITUD,...,TIPO_PROFUNDIDAD,MAGNITUD,RANGO_MAGNITUD,MC_PERIODO,SOBRE_MC,COMPARABLE_1960_2026,DEPARTAMENTO,UBICACION,DIST_LIMA_KM,DIST_COSTA_KM
0,1,1960-01-13 15:40:34,1960-01-13 10:40:34,1960,1,13,10,1960s,-16.145,-72.144,...,Superficial (0-60 km),7.5,7.0 o más,4.9,True,True,AREQUIPA,Tierra,697.6,0.0
1,2,1960-01-15 09:30:24,1960-01-15 04:30:24,1960,1,15,4,1960s,-15.000,-75.000,...,Intermedio (61-300 km),7.0,7.0 o más,4.9,True,True,ICA,Tierra,395.8,0.0
2,3,1960-01-17 02:57:58,1960-01-16 21:57:58,1960,1,16,21,1960s,-14.500,-74.500,...,Intermedio (61-300 km),6.4,6.0 - 6.9,4.9,True,True,AYACUCHO,Tierra,387.5,0.0
3,4,1960-01-23 03:37:32,1960-01-22 22:37:32,1960,1,22,22,1960s,-12.500,-68.500,...,Intermedio (61-300 km),5.8,5.0 - 5.9,4.9,True,True,FUERA DEL PERÚ,Fuera del Perú,929.5,0.0
4,5,1960-01-30 05:07:24,1960-01-30 00:07:24,1960,1,30,0,1960s,-5.500,-77.500,...,Intermedio (61-300 km),5.7,5.0 - 5.9,4.9,True,True,LORETO,Tierra,729.7,0.0
